### Varian Perlakuan Data: Deteksi & Penanganan Outlier (IQR) + Penskalaan (Seluruh Fitur)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load data bersih secara utuh
df_all = pd.read_csv('/content/data_bersih.csv')

# Pisahkan target (G3) terlebih dahulu agar tidak ikut diproses outlier/skala
X_raw = df_all.drop(columns=['G3'])
y = df_all['G3']

# Pisahkan tipe data numerik dan kategorikal dari seluruh dataset
fitur_numerik = X_raw.select_dtypes(include=[np.number]).columns.tolist()
fitur_kategorikal = X_raw.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Seluruh Fitur Numerik ({len(fitur_numerik)}): {fitur_numerik}")
print(f"Seluruh Fitur Kategorikal ({len(fitur_kategorikal)}): {fitur_kategorikal}")

# 2. Penanganan Pencilan (Outlier) dengan Metode IQR pada SEMUA Fitur Numerik
X_cleaned = X_raw.copy()
for col in fitur_numerik:
    Q1 = X_cleaned[col].quantile(0.25)
    Q3 = X_cleaned[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Membatasi (capping) nilai outlier agar berada di batas wajar
    X_cleaned[col] = np.clip(X_cleaned[col], lower_bound, upper_bound)

# 3. One-Hot Encoding untuk SEMUA fitur kategorikal
X_encoded = pd.get_dummies(
    X_cleaned,
    columns=fitur_kategorikal,
    drop_first=True
)

print(f"Total dimensi fitur setelah encoding: {X_encoded.shape[1]} kolom")

# 4. Split data TERLEBIH DAHULU sebelum scaling
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# 5. Penskalaan (Scaling) menggunakan StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Latih model & Evaluasi
model_all = LinearRegression()
model_all.fit(X_train_scaled, y_train)

y_pred = model_all.predict(X_test_scaled)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n=== HASIL VARIAN PERLAKUAN DATA (SELURUH FITUR + IQR + SCALING) ===")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")

Seluruh Fitur Numerik (15): ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2']
Seluruh Fitur Kategorikal (17): ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']
Total dimensi fitur setelah encoding: 41 kolom

=== HASIL VARIAN PERLAKUAN DATA (SELURUH FITUR + IQR + SCALING) ===
MAE  : 0.8771
RMSE : 1.3371
R2   : 0.8167
